# Punto flotante IEEE 754 y precisión de máquina

**1.** Rutina que convierte un número real a `float32` y devuelve una lista de 32 elementos.
**2.** Rutina que entrega la precisión (epsilon) del computador.

---
## 1. Conversión de un real a `float32`


| campo | bits | posición |
|---|---|---|
| signo $s$ | 1 | 0 |
| exponente  $E$ | 8 | 1–8 |
| fracción $F$ | 23 | 9–31 |


$$nim= (-1)^s \cdot \left(1 + \frac{F}{2^{23}}\right) \cdot 2^{\,E-127}$$



### Algoritmo

1. Separar el signo y trabajar con el valor absoluto
2. Escribo el numero como = m \cdot 2^e$ con $1 \le m < 2$ (lo hace `math.frexp`).
3. Sesgar el exponente: $E = e + 127$.
4. Fraccion$F = \mathrm{round}\big((m-1)\cdot 2^{23}\big)$. Se redondea porque en general
   $x$ **no** es representable exacto en 32 bits.
5. Escribir $s$, $E$ y $F$ en binario y concatenar.

In [ ]:
import math

def _int_a_bits(n, ancho):
    """Entero -> lista de bits, el más significativo primero."""
    return [(n >> i) & 1 for i in range(ancho - 1, -1, -1)]


def float32_bits(x):
    """Convierte un real a IEEE 754 float32. Devuelve una lista de 32 bits (0/1)."""
    x = float(x)
    s = 1 if x < 0 else 0 
    a = abs(x)

    # --- casos especiales -------------------------------------------------
    if math.isnan(a):
        return [s] + _int_a_bits(255, 8) + _int_a_bits(1 << 22, 23)
    if math.isinf(a):
        return [s] + _int_a_bits(255, 8) + [0] * 23
    if a == 0.0:
        return [s] + [0] * 8 + [0] * 23

    # --- normalización: a = m * 2**e  con  1 <= m < 2 ---------------------
    m, e = math.frexp(a)        # frexp entrega 0.5 <= m < 1
    m, e = m * 2, e - 1
    E = e + 127                 # exponente sesgado

    if E >= 1:                          # número normal
        F = round((m - 1.0) * 2**23)    # round() de Python redondea al par, igual que IEEE
        if F == 2**23:                  # el redondeo desbordó la mantisa: 1.111..1 -> 10.0
            F, E = 0, E + 1
    else:                               # subnormal: exponente fijo 2**-126, sin 1. implícito
        F = round(a / 2.0**-126 * 2**23)
        E = 0
        if F == 2**23:                  # el redondeo lo convirtió en el normal más chico
            F, E = 0, 1

    if E >= 255:                        # overflow -> infinito
        return [s] + _int_a_bits(255, 8) + [0] * 23

    return [s] + _int_a_bits(E, 8) + _int_a_bits(F, 23)

### Camino inverso: de los bits al número

Reconstruir el valor deja ver el **error de representación**: `0.1` y `1/3` no son
exactos en binario, así que lo que se guarda es el `float32` más cercano.

In [ ]:
def bits_a_float(bits):
    """Inversa: 32 bits -> valor real que representan."""
    s = bits[0]
    E = int(''.join(map(str, bits[1:9])), 2)
    F = int(''.join(map(str, bits[9:])), 2)

    if E == 0:                                  # cero o subnormal
        val = F * 2.0**-23 * 2.0**-126
    elif E == 255:                              # infinito o NaN
        val = math.inf if F == 0 else math.nan
    else:                                       # normal
        val = (1 + F * 2.0**-23) * 2.0**(E - 127)
    return -val if s else val


print(f"{'x original':>20} | {'x en float32':>22} | error relativo")
print("-" * 68)
for x in [0.1, 1/3, math.pi, 2.5]:
    y = bits_a_float(float32_bits(x))
    print(f"{x:>20.17g} | {y:>22.17g} | {abs(y - x)/abs(x):.3e}")

El error relativo siempre queda por debajo de $2^{-24} \approx 6\times10^{-8}$: esa es
exactamente la precisión del formato, y es de lo que trata la parte 2.

---
## 2. Precisión de la máquina

El $\varepsilon$ es el número más pequeño tal que

$$0 + \varepsilon \neq 0$$

en la aritmética del computador. 

La idea del algoritmo: partir de $\varepsilon = 1$ e ir dividiendo por 2 mientras la suma
$0+\varepsilon$ siga siendo distinta de 1. El último valor que "se nota" es el epsilon.

In [ ]:
def precision_maquina(num=0, verbose=False):
    eps = 1
    n = 0
    while num + eps / 2 != num:     # ¿la mitad todavía se nota?
        eps = eps / 2
        n += 1
        if verbose and n % 100 == 0:
            print(f"  iteración {n:3d}:  eps = {eps:.5e}")
    return eps


eps = precision_maquina(verbose=True)
print(f"comprobación:  1 + eps   != 1  ->  {1 + eps != 1}")
print(f"comprobación:  1 + eps/2 == 1  ->  {1 + eps/2 == 1}")

  iteración  10:  eps = 9.76562e-04
  iteración  20:  eps = 9.53674e-07
  iteración  30:  eps = 9.31323e-10
  iteración  40:  eps = 9.09495e-13
  iteración  50:  eps = 8.88178e-16
  iteración  60:  eps = 8.67362e-19
  iteración  70:  eps = 8.47033e-22
  iteración  80:  eps = 8.27181e-25
  iteración  90:  eps = 8.07794e-28
  iteración 100:  eps = 7.88861e-31
  iteración 110:  eps = 7.70372e-34
  iteración 120:  eps = 7.52316e-37
  iteración 130:  eps = 7.34684e-40
  iteración 140:  eps = 7.17465e-43
  iteración 150:  eps = 7.00649e-46
  iteración 160:  eps = 6.84228e-49
  iteración 170:  eps = 6.68191e-52
  iteración 180:  eps = 6.52530e-55
  iteración 190:  eps = 6.37237e-58
  iteración 200:  eps = 6.22302e-61
  iteración 210:  eps = 6.07716e-64
  iteración 220:  eps = 5.93473e-67
  iteración 230:  eps = 5.79563e-70
  iteración 240:  eps = 5.65980e-73
  iteración 250:  eps = 5.52715e-76
  iteración 260:  eps = 5.39761e-79
  iteración 270:  eps = 5.27110e-82
  iteración 280:  eps = 5.14